# Prepare training set

In [2]:
import os
import subprocess

raw_data="../../data/species"
merged_data="../results/merged"

!ls $merged_data

ls: cannot access '../results/merged': No such file or directory


## Extract CDS from annotations

In [ ]:
tr_sp=!ls $merged_data

for sp in tr_sp:
    sp=sp.replace("_own.gff", "")
    ref_ann=!realpath $merged_data/$sp*.gff
    ref_ann=ref_ann[0]
    ref_ann_name=ref_ann.split("/")[-1].replace(".gz", "").replace("_own.gff", ".gff")
    if "*" in ref_ann_name:
        print(f"No annotation for {sp}")
        continue
    print(ref_ann_name)
    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/mrg_CDS_{ref_ann_name}"
    print(result_file)

    !bash ../scripts/get_CDS.sh $ref_ann > $result_file

##grep "AP00" ../training_data/species/Cyanidioschyzon_merolae_strain_10d_280699/CDS_Cyanidioschyzon_merolae_strain_10d_280699_Ensembl_GCA_000091205.1_6ad84fe3e4d084144a801b962eb07f59.aliasMatch.gff > CDS_Cyanidioschyzon_merolae_strain_10d_280699_Ensembl_GCA_000091205.1_6ad84fe3e4d084144a801b962eb07f59.aliasMatch.filter.gff

## Sample n genes from CDS annotations

In [ ]:
tr_sp=!ls $merged_data
n=1000
#tr_sp=["Mytilus_galloprovincialis"]
for sp in tr_sp:
    sp=sp.replace("_own.gff", "")
    CDS_ann=!realpath ../training_data/species/$sp/mrg_CDS*.gff
    CDS_ann=CDS_ann[0]
    CDS_ann_name=CDS_ann.split("/")[-1]
    #skip missing gff
    if "*" in CDS_ann_name:
        print(f"No annotation for {sp}")
        continue
    print(CDS_ann_name)
    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/mrg_sample{n}_{CDS_ann_name}"
    print(result_file)

    !bash ../scripts/sample_CDS.sh $CDS_ann $n > $result_file

CDS_Babesia_duncani_323732_GenBank_GCA_028658345.1_d12006d79550f7614070864c802e7498.gff
../training_data/species/Babesia_duncani_323732/sample1000_CDS_Babesia_duncani_323732_GenBank_GCA_028658345.1_d12006d79550f7614070864c802e7498.gff
CDS_Chaetoceros_neogracilis_240364_GenBank_GCA_055048815.1_11e6046366171b1c3beb21db5e6066a1.gff
../training_data/species/Chaetoceros_neogracilis_240364/sample1000_CDS_Chaetoceros_neogracilis_240364_GenBank_GCA_055048815.1_11e6046366171b1c3beb21db5e6066a1.gff
CDS_Chlamydomonas_reinhardtii_3055_GenBank_GCA_000002595.3_86aec8f92eb0302205ae6707d5046e58.gff
../training_data/species/Chlamydomonas_reinhardtii_3055/sample1000_CDS_Chlamydomonas_reinhardtii_3055_GenBank_GCA_000002595.3_86aec8f92eb0302205ae6707d5046e58.gff
CDS_Cladocopium_goreaui_2562237_GenBank_GCA_947184155.2_a8a99f1ba09ddc5b4722bd2415c0c680.gff
../training_data/species/Cladocopium_goreaui_2562237/sample1000_CDS_Cladocopium_goreaui_2562237_GenBank_GCA_947184155.2_a8a99f1ba09ddc5b4722bd2415c0c680.g

# Training commands

In [ ]:
tr_sp=!ls $merged_data
n=1000

#tr_sp=["Mytilus_galloprovincialis"]
!mkdir -p ../job_commands
with open("../job_commands/merged_training.txt", "w") as out:
    
    #create trained parameters folder command
    parameters_folder="../results/refined_trainedParams"
    print(f"mkdir -p {parameters_folder}", file=out)
    for sp in tr_sp:
        sp=sp.replace("_own.gff", "")

        #get paths for cleaned data
        sampleN_CDS=!realpath ../training_data/species/$sp/mrg_sample$n*.gff
        sampleN_CDS=sampleN_CDS[0]
    
        clean_ref=!realpath ../training_data/species/$sp/CLEAN_*.fna
        clean_ref=clean_ref[0]

        #check if gff file is present
        if "*" in sampleN_CDS:
            print(f"No annotation for {sp}")
            continue

        #set folder for results
        result_sp=f"../results/specie_logs/{sp}/merged/"

        #training command using singularity container
        geneidtrainer_command=f"singularity run \
                            ~/software/geneidtrainerdocker.sif \
                            /scripts_geneid/geneidTRAINer4docker.pl \
                            -species {sp} \
                            -gff {sampleN_CDS} \
                            -fastas {clean_ref} \
                            -results {result_sp} \
                            -reduced no"
        geneidtrainer_command=geneidtrainer_command.replace("                             ", " ")

        #copy parameter files for clear access(+git)
        #move them to folder that will be in git, and link them to species specific
        mvParam_command=f"mv {result_sp}*.optimized.param {parameters_folder}"
        lnParam_command=f"ln -vf {parameters_folder}/{sp}.geneid.optimized.param {result_sp}"


        print(f'echo "Specie: {sp}"', file=out)
        print(geneidtrainer_command)
        print(geneidtrainer_command, file=out)
        print(mvParam_command, file=out)
        print(lnParam_command, file=out)

#@ change for python in tr_sp
#subprocess.run(["bash", "../scripts/missing_train.sh"])

#Execute generated commands to train geneId